In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_housing.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. 数据准备 ---
def get_housing_data_loaders(batch_size=128, test_size=0.2):
    """准备Ames Housing房价预测数据集"""
    logging.info("正在准备Ames Housing房价预测数据集...")

    # 加载数据
    train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
    test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

    # 合并数据集
    full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # 缺失值处理（部分特征填充为"None"或0，其他数值型用中位数）
    fill_none = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                 "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
                 "Fence", "MiscFeature", "MasVnrType"]
    for col in fill_none:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna("None")
    fill_zero = ["BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF", "GarageArea", "GarageCars", "MasVnrArea",
                 "PoolArea", "MiscVal", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch",
                 "TotRmsAbvGrd", "BedroomAbvGr", "Fireplaces", "HalfBath", "FullBath", "KitchenAbvGr",
                 "LotFrontage", "WoodDeckSF"]
    for col in fill_zero:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)
    # 其他数值型特征用中位数填充
    num_cols = full_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if full_df[col].isnull().sum() > 0:
            full_df[col] = full_df[col].fillna(full_df[col].median())

    # 删除无用特征
    cols_to_drop = ['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'LotFrontage', 'GarageYrBlt', 'MoSold', 'YrSold']
    full_df = full_df.drop(cols_to_drop, axis=1, errors='ignore')

    # 特征工程
    full_df["TotalBath"] = full_df["BsmtFullBath"] + 0.5 * full_df["BsmtHalfBath"] + full_df["FullBath"] + 0.5 * full_df["HalfBath"]
    full_df["AllSF"] = full_df["GrLivArea"] + full_df["TotalBsmtSF"]
    full_df["AllFlrsSF"] = full_df["1stFlrSF"] + full_df["2ndFlrSF"]
    full_df["AllPorchSF"] = full_df["OpenPorchSF"] + full_df["EnclosedPorch"] + full_df["3SsnPorch"] + full_df["ScreenPorch"]

    # 类别型特征独热编码
    categorical_cols = full_df.select_dtypes(include=["object"]).columns
    full_df = pd.get_dummies(full_df, columns=categorical_cols)

    # 数值型特征对偏态进行log1p变换（绝对偏度>0.5）
    from scipy.stats import skew
    numeric_feats = full_df.select_dtypes(include=[np.number]).columns
    skewed_feats = full_df[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
    skewness = skewed_feats[abs(skewed_feats) > 0.5]
    for feat in skewness.index:
        full_df[feat] = np.log1p(full_df[feat])

    # 分离训练集和测试集
    train_df = full_df.iloc[:len(train_df), :]
    test_df = full_df.iloc[len(train_df):, :].drop(['SalePrice'], axis=1)

    # 特征和目标变量
    X = train_df.drop(['SalePrice'], axis=1)
    y = train_df['SalePrice']

    # 目标变量log1p变换
    y = np.log1p(y)

    # 标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 划分训练验证集
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=42)

    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values).unsqueeze(1)  # 回归任务需要保持二维
    X_val = torch.FloatTensor(X_val)
    y_val = torch.FloatTensor(y_val.values).unsqueeze(1)

    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    logging.info("Ames Housing房价预测数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 验证样本数: {len(val_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")

    return trainloader, valloader, X_train.shape[1]

# --- 3. 模型定义 ---
class SingleLadder(nn.Module):
    def __init__(self, depth, epsilon=0.1):
        super(SingleLadder, self).__init__()
        self.depth = depth
        self.epsilon = epsilon
        
        # 可学习的权重
        self.weights = nn.Parameter(torch.randn(depth + 1) * 0.1)
        
    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs
    
    def forward(self, x_j):
        batch_size = x_j.shape[0]
        
        if self.depth == 0:
            return self.weights[0] * x_j
        
        # 从最深层开始
        result = self.weights[self.depth] * x_j
        
        for k in range(self.depth - 1, 0, -1):
            denominator = self.weights[k] * x_j + self.safe_reciprocal(result)
            result = denominator
        
        # 最后一步: w0 * x_j + 1/result
        if self.depth > 0:
            final_result = self.weights[0] * x_j + self.safe_reciprocal(result)
        else:
            final_result = self.weights[0] * x_j
            
        return final_result

class CoFrNetDRegression(nn.Module):
    def __init__(self, input_dim, max_depth=50, epsilon=0.1):
        super(CoFrNetDRegression, self).__init__()
        self.input_dim = input_dim
        self.max_depth = max_depth
        self.epsilon = epsilon
        
        # 为每个输入维度创建一个梯形
        self.ladders = nn.ModuleList()
        self.ladder_depths = []
        
        for i in range(input_dim):
            depth = min(max_depth, max(1, 5 + i % 10))  # 可调
            self.ladder_depths.append(depth)
            
            # 单个梯形
            ladder = SingleLadder(depth, epsilon)
            self.ladders.append(ladder)
        
        # 最终层 - 回归任务输出维度为1
        self.output_layer = nn.Linear(input_dim, 1)
        
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
    
    def forward(self, x):
        batch_size = x.shape[0]
        ladder_outputs = []
        
        # 计算梯形
        for j in range(self.input_dim):
            x_j = x[:, j]  
            ladder_output = self.ladders[j](x_j)
            ladder_outputs.append(ladder_output.unsqueeze(1))
        
        # 连接所有梯形输出
        ladder_features = torch.cat(ladder_outputs, dim=1)  
        
        # 最终线性组合
        output = self.output_layer(ladder_features)
        
        return output.squeeze()  # 回归任务输出一维

# --- 4. 训练函数 ---
def train_regression(model, trainloader, valloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    best_val_loss = float('inf')
    best_model_state = None
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_loss": [],
        "val_loss": []
    }

    logging.info("="*30)
    logging.info("  启动 CoFrNet-D (房价预测版) 训练流程")
    logging.info("="*30)
    
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=hparams['patience'], factor=0.5)
    
    num_epochs = hparams['num_epochs']
    patience = hparams['patience']
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        epoch_avg_loss = running_loss / len(trainloader)
        log_data["training_loss"].append(epoch_avg_loss)
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item()
        
        avg_val_loss = val_loss / len(valloader)
        log_data["val_loss"].append(avg_val_loss)
        scheduler.step(avg_val_loss)
        
        logging.info(f"Epoch {epoch+1}/{num_epochs} | 训练MSE损失: {epoch_avg_loss:.4f} | 验证MSE损失: {avg_val_loss:.4f}")
        
        # 早停检查
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict()
            logging.info(f"*** 新的最佳MSE损失: {best_val_loss:.4f} ***")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logging.info(f"早停触发 - 验证损失在{patience}个epoch内未改善")
                break
    
    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_val_loss,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出任何有效模型，不进行保存。")
    
    with open(log_filepath, 'w') as f:
        json.dump(log_data, f, indent=4)
    
    return best_val_loss, best_model_state

# --- 5. 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_housing.log'
    JSON_LOG_FILE = 'training_housing_log.json'
    MODEL_SAVE_PATH = 'housing_cofrnet_d.pth'
    
    HYPERPARAMETERS = {
        "num_epochs": 200,
        "learning_rate": 0.001,
        "weight_decay": 1e-5,
        "patience": 20,
        "max_depth": 30,
        "epsilon": 0.1
    }

    setup_logging(LOG_FILE)
    logging.info(f"房价预测实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, valloader, input_dim = get_housing_data_loaders()
    
    # 创建CoFrNet-D回归模型
    cofr_net = CoFrNetDRegression(
        input_dim=input_dim,
        max_depth=HYPERPARAMETERS['max_depth'],
        epsilon=HYPERPARAMETERS['epsilon']
    ).to(device)

    start_time = time.time()
    best_loss, best_state_dict = train_regression(
        cofr_net, trainloader, valloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    
    # 加载并验证保存的最佳模型
    if best_state_dict:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳房价预测模型")
        logging.info("="*30)
        
        verification_model = CoFrNetDRegression(
            input_dim=input_dim,
            max_depth=HYPERPARAMETERS['max_depth'],
            epsilon=HYPERPARAMETERS['epsilon']
        )
        verification_model.load_state_dict(best_state_dict)
        verification_model.to(device)
        verification_model.eval()

        logging.info(f"成功加载模型。保存的MSE损失: {best_loss:.4f}")

        total_params = sum(p.numel() for p in verification_model.parameters() if p.requires_grad)
        logging.info(f"模型总参数数量: {total_params}")

        val_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                val_loss += criterion(outputs, targets).item()
        
        avg_val_loss = val_loss / len(valloader)
        logging.info(f"加载后的模型在验证集上的MSE损失: {avg_val_loss:.4f}")
        
        assert abs(avg_val_loss - best_loss) < 1e-3, "验证失败！"
        logging.info("验证成功！房价预测模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存房价预测模型。")

2025-08-14 12:14:37,657 [INFO ]  房价预测实验超参数: {
    "num_epochs": 200,
    "learning_rate": 0.001,
    "weight_decay": 1e-05,
    "patience": 20,
    "max_depth": 30,
    "epsilon": 0.1
}
2025-08-14 12:14:37,658 [INFO ]  正在准备Ames Housing房价预测数据集...
2025-08-14 12:14:37,988 [INFO ]  Ames Housing房价预测数据集准备完毕。
2025-08-14 12:14:37,989 [INFO ]  训练样本数: 1168, 验证样本数: 292
2025-08-14 12:14:37,990 [INFO ]  特征维度: 279, 目标维度: 1
2025-08-14 12:14:38,033 [INFO ]  ==============================
2025-08-14 12:14:38,034 [INFO ]    启动 CoFrNet-D (房价预测版) 训练流程
2025-08-14 12:14:38,034 [INFO ]  ==============================


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([128, 1])) that is different to the input size (torch.Size([128])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([16, 1])) that is different to the input size (torch.Size([16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


2025-08-14 12:14:47,215 [INFO ]  Epoch 1/200 | 训练MSE损失: 33.4622 | 验证MSE损失: 11.0599
2025-08-14 12:14:47,218 [INFO ]  *** 新的最佳MSE损失: 11.0599 ***


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([36, 1])) that is different to the input size (torch.Size([36])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


2025-08-14 12:14:52,952 [INFO ]  Epoch 2/200 | 训练MSE损失: 10.2003 | 验证MSE损失: 10.3950
2025-08-14 12:14:52,955 [INFO ]  *** 新的最佳MSE损失: 10.3950 ***
2025-08-14 12:14:58,737 [INFO ]  Epoch 3/200 | 训练MSE损失: 18.8523 | 验证MSE损失: 16.9826
2025-08-14 12:15:04,504 [INFO ]  Epoch 4/200 | 训练MSE损失: 11.6755 | 验证MSE损失: 44.4218
2025-08-14 12:15:10,315 [INFO ]  Epoch 5/200 | 训练MSE损失: 36.7029 | 验证MSE损失: 6.2965
2025-08-14 12:15:10,319 [INFO ]  *** 新的最佳MSE损失: 6.2965 ***
2025-08-14 12:15:16,205 [INFO ]  Epoch 6/200 | 训练MSE损失: 11.5374 | 验证MSE损失: 82.2665
2025-08-14 12:15:21,764 [INFO ]  Epoch 7/200 | 训练MSE损失: 34.8622 | 验证MSE损失: 16.4225
2025-08-14 12:15:27,520 [INFO ]  Epoch 8/200 | 训练MSE损失: 12.0017 | 验证MSE损失: 19.0953
2025-08-14 12:15:33,384 [INFO ]  Epoch 9/200 | 训练MSE损失: 26.0027 | 验证MSE损失: 24.9587
2025-08-14 12:15:38,940 [INFO ]  Epoch 10/200 | 训练MSE损失: 10.9226 | 验证MSE损失: 7.3746
2025-08-14 12:15:44,893 [INFO ]  Epoch 11/200 | 训练MSE损失: 7.0885 | 验证MSE损失: 3.6879
2025-08-14 12:15:44,896 [INFO ]  *** 新的最佳MSE损失: 3.687

AssertionError: 验证失败！